# 01 · The intraday close regime — mechanism & how it was found

*Edge-features arc · 01 discovery · 02 linear base · 03 hero cascade · 04 regime-MoE · 05 temporal-kNN  —  machinery: `ridge_pipeline_throughline.ipynb`*

**Verify first / interpret second.** This chapter is the *why*: what the residualized-EBM campaign
actually learned, and the falsifiable loop that surfaced it. The discovery numbers are **cluster**
walk-forwards (full-OOS Duan-smeared QLIKE) produced by small scripts that live cluster-side at
`/scratch1/jc_905/harxhar-clean` (named in `writeup/intraday_regime_findings_2026-06-26.md`
§Reproducibility) — **the prediction data is not in this repo**, so every number below is shown **with
its exact reproduce-command** (the script + args that emit it), never a bare paste and never a faked
local recompute. The local machinery the discovery is *about* (the HAR×{open,close} regime columns,
the close gate) is folded in full from `resid_amortized.py`.

The headline: the residual the tuned tree extracts is an **intraday auction/session-transition regime** —
HAR volatility-persistence **sign-flips at the session edges** — not a leftover diurnal U-shape.

In [ ]:
import html, inspect, os, sys, textwrap
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
pd.set_option("display.max_colwidth", None)

def find_repo(s):
    for q in [Path(s).resolve(), *Path(s).resolve().parents]:
        if (q / "resid_amortized.py").exists() and (q / "src").is_dir():
            return q
    raise FileNotFoundError("repo root")
REPO = find_repo(Path.cwd()); os.chdir(REPO); sys.path.insert(0, str(REPO))

# Collapsible, theme-following source display (one <details> per function; folded) — the same
# helper as notebooks/results/bucket_sweep.ipynb.
def _details(f, open_=False):
    path = f.__module__.replace("src.", "src/").replace(".", "/") + ".py"
    try:
        sig = ("class " + f.__name__) if inspect.isclass(f) else ("def " + f.__name__ + str(inspect.signature(f)))
    except (ValueError, TypeError):
        sig = f.__qualname__
    body = "```python\n" + textwrap.dedent(inspect.getsource(f)).rstrip() + "\n```"
    return (f"<details{' open' if open_ else ''}>\n<summary><code>{html.escape(path + '  ·  ' + sig)}"
            f"</code></summary>\n\n{body}\n\n</details>")
def show_one(f): return Markdown(_details(f))   # thin: just this function (no transitive tree)

from resid_amortized import _regime_interactions, _close_mask, _cumrv_close, CACHE_ROOT

# "No bare number" invariant: every quantitative row must carry a non-empty reproduce-command.
def assert_backed(df, col="reproduce"):
    miss = df[df[col].astype(str).str.strip() == ""]
    assert miss.empty, f"unbacked numbers (no reproduce-command): {miss.index.tolist()}"
    return len(df)

PY = "$PY"  # cluster python (conda env 285J at /scratch1/jc_905/harxhar-clean); the scripts run THERE.
            # This repo carries neither them nor the prediction cache, so numbers are shown with their
            # reproduce-command, never recomputed locally from data that isn't present.
print("setup ok | local resid_prep cells:",
      sorted(p.name for p in Path(CACHE_ROOT).glob("*")) if Path(CACHE_ROOT).is_dir() else "(none)")

---
## 1 · The discovery loop (the asset)

The close regime was found by a **fixed 6-step loop**, run once on the `hour` axis
(`writeup/discovery_process_methodology_2026-06-29.md`). Each step's tool is a small cluster-side
script; the table pairs every number with the **exact command that regenerates it**. The scripts are
**not vendored here** (cluster-side at `/scratch1/jc_905/harxhar-clean`), so these are cluster values,
reproduce-backed — not local recomputes.

In [ ]:
loop = pd.DataFrame([
    (1, "black box beats the linear base",
        "EBM-on-residual 0.12414 vs plain-enet 0.12516 → the residual HAS structure",
        f"{PY} resid_amortized.py chunk_collect ebm_all_buckets_tw1000_enet_rf480_slim resid_subset   # 0.12414"
        f"  ||  {PY} resid_amortized.py prep xgb all_buckets 1000 1 480 slim enet   # base_alone_qlike=0.12516"),
    (2, "read the black box",
        "hour: enet |coef| rank 105 → EBM rank 7; 3/5 top interactions involve hour",
        f"{PY} ebm_interpret.py ebm_all_buckets_tw1000_enet_rf480_slim   # main-effect + pairwise importance"),
    (3, "test the interaction OOS",
        "clock-regime +0.0095 OOS R²; vol-state −0.0027 (fails); HAR persistence flips 5/6 windows",
        f"{PY} regime_study.py   # predictor×regime OOS R2   ||   {PY} har_flip.py   # rolling persistence sign"),
    (4, "localize sharply (the decisive plot)",
        "corr(har_ma_5, resid) by hour: −0.085 @ h9 (open), −0.21 @ h17 (close/AH); sharp, not gradual",
        f"{PY} tests123.py   # corr(har_ma_5, resid) bucketed by clock hour"),
    (5, "distill to one feature",
        "HAR × late-day recovers 58% of the tree's edge",
        f"{PY} distill_regime.py   # fit HAR×late-day, measure edge recovered"),
    (6, "falsification gauntlet",
        "diurnal-U control survives; gamma weak (~20%); real-space; FORCE past L1 → ~38% genuinely higher-order",
        f"{PY} regime_study.py --control diurnalU   ||   FORCE_COLS=... {PY} resid_amortized.py trial <cell> resid_subset"),
], columns=["#", "step", "output (cluster value)", "reproduce"])
assert_backed(loop)
need = ["ebm_interpret.py", "regime_study.py", "har_flip.py", "tests123.py", "distill_regime.py"]
present = [s for s in need if (REPO / s).exists()]
ebm_cache = sorted(p.name for p in Path(CACHE_ROOT).glob("*ebm*")) if Path(CACHE_ROOT).is_dir() else []
display(loop.set_index("#"))
print(f"PASS — all {len(loop)} discovery numbers carry a reproduce-command.")
print(f"  discovery scripts vendored locally: {present or '(none — cluster-side at /scratch1/jc_905/harxhar-clean)'}")
print(f"  prediction cache present locally  : {ebm_cache or '(none — cluster-side; numbers shown, not recomputed)'}")

**The object of the discovery, folded from source.** The scripts are cluster-side, but the thing they
localized — the HAR×{open,close} session-edge interaction columns, the close/after-hours gate, and the
intraday vol-path accumulation — is real, vendored machinery in `resid_amortized.py`. Step 4 is literally
`corr(har_ma_5, resid)` inside the `_close_mask` window; step 5's distilled feature is one of these
`_regime_interactions` columns. Folded in full:

In [ ]:
display(show_one(_regime_interactions))
display(show_one(_close_mask))
display(show_one(_cumrv_close))

## 2 · The compass — the tree-subsumption law

A boosted tree / EBM is **invariant to monotone transforms of the current row**. So the *only* features
that beat a fitted tree are **functionals of history/sequence the row doesn't contain**. This tells you
where to look (history-dependent regime functionals) and when you're done (when the only surviving lever
is 5th-decimal → the lever is *data*, not features). It also frames everything downstream: the linear
improvers (ch. 02) get **absorbed** by the tree; the regime stage (ch. 03) is what survives.

### Mechanism numbers — each reproduce-backed (the interpret below reads these)

The interpret section's coefficients and edges are not pasted bare either; each is the printed output of
a named build (`enetreg2_harunpen` prints the legible HAR×gate coefs as `CANON_COEFS`; `fwl_attribution.py`
reproduces the `cumrv×close` edge — the full block decomposition is ch. 02).

In [ ]:
mech = pd.DataFrame([
    ("har_ma_5×close coef = −0.05 (close DAMPS short-horizon persistence)",
        f"{PY} resid_amortized.py prep xgb all_buckets 1000 1 480 slim enetreg2_harunpen   # CANON_COEFS har_ma_5_x_close"),
    ("har_ma_1×open coef = −0.045 (open damps the 1-bar)",
        f"{PY} resid_amortized.py prep xgb all_buckets 1000 1 480 slim enetreg2_harunpen   # CANON_COEFS har_ma_1_x_open"),
    ("cumrv×close edge −0.00122 (base 0.12436 → 0.12314)",
        f"{PY} fwl_attribution.py   # Type-III −CUMRV row; FULL fit reproduces 0.12314"),
    ("sqrt vol-scale accumulation = 97.5% of the cumrv gap",
        f"{PY} resid_amortized.py prep xgb all_buckets 1000 1 480 slim enetreg2_realsqrt   # cumsum(sqrt(adj RV)) isolate"),
    ("dealer-gamma ~20% modulator (secondary; clock dominates)",
        f"{PY} regime_study.py   # voldemand×regime in-region corr"),
], columns=["mechanism claim", "reproduce"])
assert_backed(mech)
display(mech)

## 3 · Interpret — a session-edge vol regime

- **The close DAMPS short-horizon persistence** — read off the legible unpenalized-HAR base:
  `har_ma_5×close` coef = **−0.05** (high recent vol → close forecast pulled *below* HAR's extrapolation).
- **The intraday vol PATH matters at the close** — `cumrv×close` is the single biggest engineered edge
  (base **0.12436 → 0.12314, −0.00122**); mechanistically the **sqrt vol-scale accumulation** (97.5% of it).
- **The open is a discrete auction/gap event** — a symmetric overnight-cumrv-at-open feature is null.
- **Clock-anchored, not vol-state-anchored**; dealer-gamma is a weak (~20%) modulator.

⇒ the residual edge is the equilibrium footprint of auction / MOC liquidity + dealer hedging — tiny,
already-arbitraged, strongest where costs are highest. The deliverable is the **mechanism + a map of
which data to buy** (auction imbalance / GEX / OFI), not the 4th-decimal QLIKE. Continues in **ch. 02**.